# 04 — Model Interpretation & Error Analysis

**Project:** Hospital Readmission Prediction  
**Input:** `models/best_tuned_model.pkl` (calibrated model + threshold)

| # | Section |
|---|---|
| 1 | Environment setup |
| 2 | Load model + features |
| 3 | SHAP global importance (bar chart) |
| 4 | SHAP beeswarm summary |
| 5 | SHAP dependence plots (top features) |
| 6 | SHAP waterfall — individual examples |
| 7 | Error group classification |
| 8 | Error group summary statistics |
| 9 | Error distributions (violin plots) |
| 10 | FP vs FN profile comparison |
| 11 | Diagnosis distribution by error group |
| 12 | Takeaways |

---
## 1 — Environment setup

In [1]:
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings('ignore')

ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.utils import load_config, get_logger, set_seed, load_model

config = load_config(ROOT / 'config' / 'config.yaml')
set_seed(config['random_seed'])
config['_base_dir'] = str(ROOT)
config['_figures_subfolder'] = 'shap'

logger = get_logger('04_interpretation')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

print(f'ROOT: {ROOT}')
print('Environment ready.')

ROOT: C:\Users\speak\HA_Project\Hospital-Readmission-Project-
Environment ready.


---
## 2 — Load model + features

In [2]:
from src.modeling import load_features, make_splits

X, y, feature_names = load_features(config, base_dir=ROOT)
df_analysis = pd.read_csv(ROOT / config["paths"]["analysis_features_data"], index_col="row_id")
X_train, X_val, X_test, y_train, y_val, y_test = make_splits(X, y, config)
X_val_raw = df_analysis.loc[X_val.index]

print(f'Features: {X.shape[1]} | Val: {len(y_val):,} | Test: {len(y_test):,}')


2026-03-11 23:02:46 | INFO     | src.modeling | Loaded feature dataset: 25000 rows × 48 columns


2026-03-11 23:02:46 | INFO     | src.modeling | Feature matrix: 25000 rows × 47 features | pos=11754 (47.0%) neg=13246 (53.0%)


2026-03-11 23:02:46 | INFO     | src.modeling | Split: train=15000 | val=5000 | test=5000


Features: 47 | Val: 5,000 | Test: 5,000


In [3]:
# Load saved calibrated model
model_artifact = load_model(ROOT / config['paths']['model_dir'] / 'best_tuned_model.pkl')

if isinstance(model_artifact, dict):
    model     = model_artifact['model']
    threshold = model_artifact['threshold']
    model_name = model_artifact.get('model_name', 'BestTunedModel')
else:
    # Backward-compatible: artifact is the model itself
    model     = model_artifact
    threshold = 0.5
    model_name = 'BestTunedModel'

print(f'Model     : {model_name}')
print(f'Threshold : {threshold:.3f}')
print(f'Model type: {type(model).__name__}')

Model     : GradientBoosting
Threshold : 0.350
Model type: _PrefitCalibratedModel


---
## 3 — SHAP global importance (bar chart)

SHAP values are computed via `shap.Explainer` on `model.predict_proba` — explaining the **full calibrated model** including any scaling. Feature values reflect the original (unscaled) input space.

In [4]:
from src.interpretation import compute_shap_values, plot_shap_bar

shap_values, X_sample = compute_shap_values(model, X_val, config, max_samples=500)
print(f'SHAP shape: {shap_values.values.shape}')
print(f'Sample size: {len(X_sample):,}')

2026-03-11 23:02:47 | INFO     | src.interpretation | Computing SHAP values for 500 samples via model-agnostic Explainer...


PermutationExplainer explainer:   8%|▊         | 41/500 [00:00<?, ?it/s]

PermutationExplainer explainer:   9%|▉         | 44/500 [00:10<00:15, 28.76it/s]

PermutationExplainer explainer:   9%|▉         | 47/500 [00:10<00:22, 20.05it/s]

PermutationExplainer explainer:  10%|█         | 50/500 [00:10<00:25, 17.88it/s]

...

PermutationExplainer explainer: 100%|█████████▉| 499/500 [00:38<00:00, 15.55it/s]

PermutationExplainer explainer: 501it [00:38, 15.75it/s]                         

PermutationExplainer explainer: 501it [00:38, 12.04it/s]


2026-03-11 23:03:26 | INFO     | src.interpretation | SHAP values computed. Shape: (500, 47)


SHAP shape: (500, 47)
Sample size: 500


In [5]:
plot_shap_bar(shap_values, config, max_features=20)

2026-03-11 23:03:26 | INFO     | src.interpretation | Saved SHAP bar importance → C:\Users\speak\HA_Project\Hospital-Readmission-Project-\reports\figures\shap\shap_bar_importance.png


---
## 4 — SHAP beeswarm summary

In [6]:
from src.interpretation import plot_shap_summary

plot_shap_summary(shap_values, X_sample, config, max_features=20)

2026-03-11 23:03:27 | INFO     | src.interpretation | Saved SHAP summary → C:\Users\speak\HA_Project\Hospital-Readmission-Project-\reports\figures\shap\shap_summary.png


---
## 5 — SHAP dependence plots (top 4 features)

Dependence plots show how a feature's value affects its SHAP contribution. Axes show real feature values (not z-scores) since SHAP explains the model via unscaled inputs.

In [7]:
from src.interpretation import plot_shap_dependence

plot_shap_dependence(shap_values, X_sample, config, top_n=4)

2026-03-11 23:03:27 | INFO     | src.interpretation | Saved SHAP dependence (total_prior_utilization) → C:\Users\speak\HA_Project\Hospital-Readmission-Project-\reports\figures\shap\shap_dependence_total_prior_utilization.png


2026-03-11 23:03:27 | INFO     | src.interpretation | Saved SHAP dependence (n_inpatient_x_time_in_hospital) → C:\Users\speak\HA_Project\Hospital-Readmission-Project-\reports\figures\shap\shap_dependence_n_inpatient_x_time_in_hospital.png


2026-03-11 23:03:28 | INFO     | src.interpretation | Saved SHAP dependence (diabetes_med_yes) → C:\Users\speak\HA_Project\Hospital-Readmission-Project-\reports\figures\shap\shap_dependence_diabetes_med_yes.png


2026-03-11 23:03:28 | INFO     | src.interpretation | Saved SHAP dependence (age_ordinal) → C:\Users\speak\HA_Project\Hospital-Readmission-Project-\reports\figures\shap\shap_dependence_age_ordinal.png


---
## 6 — SHAP waterfall — individual examples

In [8]:
from src.interpretation import plot_shap_waterfall

# Set subfolder for SHAP plots
config['_figures_subfolder'] = 'shap'
plot_shap_waterfall(shap_values, config, n_examples=3)

2026-03-11 23:03:28 | INFO     | src.interpretation | Saved SHAP waterfall (example 1) → C:\Users\speak\HA_Project\Hospital-Readmission-Project-\reports\figures\shap\shap_patient_example_1.png


2026-03-11 23:03:29 | INFO     | src.interpretation | Saved SHAP waterfall (example 2) → C:\Users\speak\HA_Project\Hospital-Readmission-Project-\reports\figures\shap\shap_patient_example_2.png


2026-03-11 23:03:29 | INFO     | src.interpretation | Saved SHAP waterfall (example 3) → C:\Users\speak\HA_Project\Hospital-Readmission-Project-\reports\figures\shap\shap_patient_example_3.png


---
## 7 — Error group classification

Classify val-set predictions into TP / TN / FP / FN at the optimal threshold.

In [9]:
from src.interpretation import compute_error_groups

# Switch to error_analysis subfolder
config['_figures_subfolder'] = 'error_analysis'

groups = compute_error_groups(model, X_val, y_val, threshold=threshold)

for name, idx in groups.items():
    print(f'{name}: {len(idx):,} rows ({len(idx)/len(y_val)*100:.1f}%)')

2026-03-11 23:03:29 | INFO     | src.interpretation | Error group TP  : 2061 rows (41.2%)


2026-03-11 23:03:29 | INFO     | src.interpretation | Error group TN  : 684 rows (13.7%)


2026-03-11 23:03:29 | INFO     | src.interpretation | Error group FP  : 1965 rows (39.3%)


2026-03-11 23:03:29 | INFO     | src.interpretation | Error group FN  : 290 rows (5.8%)


TP: 2,061 rows (41.2%)
TN: 684 rows (13.7%)
FP: 1,965 rows (39.3%)
FN: 290 rows (5.8%)


---
## 8 — Error group summary statistics

In [10]:
from src.interpretation import summarise_errors

# Use X_val_raw (pre-OHE) for error summaries
error_summary = summarise_errors(groups, X_val_raw, config)
display(error_summary.round(3))


2026-03-11 23:03:29 | INFO     | src.interpretation | Error summary:
          n  mean_age_ordinal  mean_time_in_hospital  mean_n_medications  mean_n_lab_procedures  mean_n_inpatient  mean_n_emergency  mean_total_prior_utilization
group                                                                                                                                                            
TP     2061          3.476953               4.643862           16.803493              44.563319          0.998059          0.286754                      1.829209
TN      684          2.603801               3.421053           14.384503              39.361111          0.004386          0.007310                      0.038012
FP     1965          3.547074               4.594911           16.598982              43.811196          0.499237          0.133842                      0.973028
FN      290          2.575862               3.796552           15.296552              41.582759          0.017241        

,n,mean_age_ordinal,mean_time_in_hospital,mean_n_medications,mean_n_lab_procedures,mean_n_inpatient,mean_n_emergency,mean_total_prior_utilization
group,,,,,,,,
TP,2061,3.477,4.644,16.803,44.563,0.998,0.287,1.829
TN,684,2.604,3.421,14.385,39.361,0.004,0.007,0.038
FP,1965,3.547,4.595,16.599,43.811,0.499,0.134,0.973
FN,290,2.576,3.797,15.297,41.583,0.017,0.003,0.028


---
## 9 — Error distributions (violin plots)

In [11]:
from src.interpretation import plot_error_distributions

plot_error_distributions(groups, X_val_raw, config)


2026-03-11 23:03:30 | INFO     | src.interpretation | Saved error distribution (age_ordinal) → C:\Users\speak\HA_Project\Hospital-Readmission-Project-\reports\figures\error_analysis\error_age_ordinal_distribution.png


2026-03-11 23:03:30 | INFO     | src.interpretation | Saved error distribution (time_in_hospital) → C:\Users\speak\HA_Project\Hospital-Readmission-Project-\reports\figures\error_analysis\error_time_in_hospital_distribution.png


2026-03-11 23:03:30 | INFO     | src.interpretation | Saved error distribution (n_medications) → C:\Users\speak\HA_Project\Hospital-Readmission-Project-\reports\figures\error_analysis\error_n_medications_distribution.png


2026-03-11 23:03:30 | INFO     | src.interpretation | Saved error distribution (n_inpatient) → C:\Users\speak\HA_Project\Hospital-Readmission-Project-\reports\figures\error_analysis\error_n_inpatient_distribution.png


2026-03-11 23:03:30 | INFO     | src.interpretation | Saved error distribution (total_prior_utilization) → C:\Users\speak\HA_Project\Hospital-Readmission-Project-\reports\figures\error_analysis\error_total_prior_utilization_distribution.png


---
## 10 — FP vs FN profile comparison

In [12]:
from src.interpretation import plot_false_positive_vs_negative

plot_false_positive_vs_negative(groups, X_val_raw, config)


2026-03-11 23:03:31 | INFO     | src.interpretation | Saved FP vs FN comparison → C:\Users\speak\HA_Project\Hospital-Readmission-Project-\reports\figures\error_analysis\false_positive_vs_negative.png


---
## 11 — Diagnosis distribution by error group

Primary diagnosis (`diag_1`) breakdown for false positives vs false negatives.

In [13]:
from src.interpretation import plot_error_diagnosis_distribution

plot_error_diagnosis_distribution(groups, X_val_raw, config, diag_col='diag_1')


2026-03-11 23:03:31 | WARNING  | src.interpretation | 'diag_1' not in df — skipping diagnosis plot.


---
## 12 — Age vs utilisation scatter

In [14]:
from src.interpretation import plot_error_utilization_scatter

plot_error_utilization_scatter(groups, X_val_raw, config)


2026-03-11 23:03:31 | INFO     | src.interpretation | Saved age-utilization scatter → C:\Users\speak\HA_Project\Hospital-Readmission-Project-\reports\figures\error_analysis\error_age_utilization_scatter.png


---
## 13 - Takeaways

### SHAP findings
- SHAP was computed via model-agnostic `shap.Explainer` on the **full calibrated model** (`_PrefitCalibratedModel.predict_proba`).
- Feature values in dependence plots reflect real clinical units (e.g. `time_in_hospital`: 1-14 days).
- Key drivers are visible from the beeswarm and bar plots.

### Error analysis findings
- With balanced classes (~47/53), all four error groups (TP/TN/FP/FN) contain meaningful sample sizes.
- FP profile vs FN profile reveals which patient subgroups are harder to classify correctly.
- Diagnosis distribution identifies if specific primary diagnoses are over-represented in errors.

### Limitations
- SHAP explanations are approximate for calibrated models (calibration layer is treated as part of the black box).
- Error analysis is on the validation set - generalization to test set not guaranteed.
- This repo documents one checked-in dataset; generalization to other hospital systems is unverified.